In [0]:

%sql
INSERT INTO workspace.logs.master_log (JobName, StartTime, Status)
VALUES ('Training_Job', current_timestamp(), 'RUNNING');

In [0]:
%sql

-- CHILD START: log the Staging notebook
WITH active_run AS (
  SELECT MasterLogId
  FROM workspace.logs.master_log
  WHERE JobName = 'Training_Job' AND Status = 'RUNNING'
  ORDER BY MasterLogId DESC
  LIMIT 1
)
INSERT INTO workspace.logs.child_log (MasterLogId, NotebookName, StartTime, Status)
SELECT MasterLogId, 'Staging', current_timestamp(), 'RUNNING'
FROM active_run;


In [0]:
from pyspark.sql import SparkSession,DataFrame
from pyspark.sql.functions import col,struct,collect_list,lower,to_json,explode_outer,current_timestamp,current_date
from pyspark.sql.types import StructType,StringType, TimestampType, LongType,IntegerType,ArrayType,MapType,StructField

In [0]:
CATALOG_NAME = "workspace"
CONTROL_SCHEMA = "controlschema"
STAGING_SCHEMA = "staging"
FILE_LOG_TABLE = f"{CATALOG_NAME}.{CONTROL_SCHEMA}.File_Log"
STAGING_PATH = f"{CATALOG_NAME}.{STAGING_SCHEMA}"


In [0]:


query = f"""
SELECT
  f.FileID,
  f.SourceFileName,
  f.SourceFilePath,
  f.TargetTableName,
  collect_list(
    named_struct(
      'SourceColumnName', s.SourceColumnName,
      'TargetColumnName', s.TargetColumnName,
      'TargetDataType',  s.TargetDataType,
      'SourceDataType',  s.SourceDataType
    )
  ) AS columns
FROM `{CATALOG_NAME}`.`{CONTROL_SCHEMA}`.`File_Details` AS f
INNER JOIN `{CATALOG_NAME}`.`{CONTROL_SCHEMA}`.`File_Schema_Details` AS s
  ON f.FileID = s.FileID
WHERE f.IsActive = 'Y'
  AND s.IsActive = 'Y'
GROUP BY
  f.FileID,
  f.SourceFileName,
  f.SourceFilePath,
  f.TargetTableName
"""
active_files_df = spark.sql(query)
display(active_files_df)

 

In [0]:
#Keep only CSV files 
csv_rows = (
    active_files_df
        .filter(lower(col("SourceFileName")).endswith(".csv"))
        .collect()   
)

log_entries = log_entries if 'log_entries' in globals() else []

for row in csv_rows:
    file_id      = row["FileID"]
    file_name    = row["SourceFileName"]
    source_path  = row["SourceFilePath"]
    target_table = row["TargetTableName"]
    columns_list = row["columns"] 

    load_status = "SUCCESS"
    row_count   = 0
    fq_table = f"{CATALOG_NAME}.{STAGING_SCHEMA}.{target_table}"
    TARGET_TABLE_PATH = f"{STAGING_PATH}.{target_table}"
    if file_name.endswith(".csv"):
        try:
            sql_query = (f"""
                CREATE OR REPLACE TABLE {fq_table}
                USING DELTA
                AS SELECT {", ".join([c[0] for c in columns_list])}
                FROM read_files(
                    '{source_path}',
                    format => 'csv',
                    header => true,
                    inferSchema => true
                )
            """)
            print(STAGING_PATH)
            spark.sql(sql_query)
            row_count = spark.table(fq_table).count()
            load_status = "SUCCESS"
            print(f"[CSV] Success for {file_name} at {fq_table}")
        except Exception as e:
                load_status = f"FAILED: {str(e)}"
                print(f"[CSV] Error for {file_name}: {load_status}")
        #append log entry
        spark.sql(f"""
            INSERT INTO {FILE_LOG_TABLE}
            VALUES ('{file_id}', '{file_name}', current_timestamp(),"{load_status}",{row_count})
        """)
    else:
        print(f"{file_name} is json,Run the next cell")



In [0]:

def flatten_json_df(df: active_files_df, explode_arrays: bool = True, separator: str = "_") -> DataFrame:
    """
    Recursively flattens a PySpark DataFrame containing nested JSON structures.
    
    - StructType: expands nested fields to top-level columns using 'parent{separator}child' naming.
    - ArrayType:
        * explode_arrays=True: explodes one array per iteration into multiple rows.
        * explode_arrays=False: keeps arrays as JSON strings (one row per input record).
    - MapType: converts maps to JSON strings (to avoid generating dynamic columns).
    
    Args:
        df: Spark DataFrame to flatten.
        explode_arrays: Whether to explode arrays into multiple rows.
        separator: Separator used when naming flattened struct fields.
    Returns:
        Flattened Spark DataFrame.
    """
    current_df = df

    while True:
        fields = current_df.schema.fields
        struct_cols = [f for f in fields if isinstance(f.dataType, StructType)]
        array_cols  = [f for f in fields if isinstance(f.dataType, ArrayType)]
        map_cols    = [f for f in fields if isinstance(f.dataType, MapType)]

        # Convert maps to JSON string to avoid generating columns from arbitrary keys
        if map_cols:
            for f in map_cols:
                current_df = current_df.withColumn(f.name, to_json(col(f.name)))
            # re-scan after map conversions
            fields = current_df.schema.fields
            struct_cols = [f for f in fields if isinstance(f.dataType, StructType)]
            array_cols  = [f for f in fields if isinstance(f.dataType, ArrayType)]

        # stop when nothing complex remains
        if not struct_cols and not array_cols:
            break

        # 1) Flatten all struct columns in one pass
        if struct_cols:
            select_cols = [col(f.name) for f in fields if not isinstance(f.dataType, StructType)]
            for f in struct_cols:
                parent = f.name
                for sf in f.dataType.fields:
                    select_cols.append(col(f"{parent}.{sf.name}").alias(f"{parent}{separator}{sf.name}"))
            current_df = current_df.select(*select_cols)
            # continue to next loop to check for remaining arrays
            continue

        # 2) Handle arrays
        arr_field = array_cols[0]
        arr_name  = arr_field.name

        if explode_arrays:
            # explode one array per iteration; use explode_outer to preserve null/empty arrays as null rows
            current_df = current_df.withColumn(arr_name, explode_outer(col(arr_name)))
            # next iteration will flatten any struct produced by the explode
            continue
        else:
            # keep one row per record: convert arrays to JSON string
            current_df = current_df.withColumn(arr_name, to_json(col(arr_name)))
            # re-scan after conversion (array becomes string)
            continue

    return current_df

In [0]:


# Keep only JSON files
json_rows = (
    active_files_df
        .filter(lower(col("SourceFileName")).endswith(".json"))
        .collect()
)

log_entries = log_entries if 'log_entries' in globals() else []

for row in json_rows:
    file_id      = row["FileID"]
    file_name    = row["SourceFileName"]
    source_path  = row["SourceFilePath"]
    target_table = row["TargetTableName"]
    columns_list = row["columns"]   

    load_status = "SUCCESS"
    row_count   = 0

    fq_table = f"{CATALOG_NAME}.{STAGING_SCHEMA}.{target_table}"
    TARGET_TABLE_PATH = f"{STAGING_PATH}.{target_table}"

    if file_name.lower().endswith(".json"):
        try:
            # 1) Read raw JSON via SQL (no views used; this returns a DataFrame)
            raw_df = spark.sql(f"""
                SELECT {", ".join([c[0] for c in columns_list])}
                FROM read_files(
                    '{source_path}',
                    format      => 'json',
                    inferSchema => true,
                    multiLine   => true
                )
            """)

            # 2) Flatten using your function 
            flat_df = flatten_json_df(raw_df)
           
            # 4) Write to staging
            (flat_df.write.mode("overwrite").format("delta").saveAsTable(fq_table))

            # 5) Row count & status
            row_count = spark.table(fq_table).count()
            load_status = "SUCCESS"
            print(f"[JSON] Success for {file_name} at {fq_table}")

        except Exception as e:
            load_status = f"FAILED: {str(e)}"
            print(f"[JSON] Error for {file_name}: {load_status}")

        # 6) Append log entry 
        spark.sql(f"""
            INSERT INTO {FILE_LOG_TABLE}
            VALUES ('{file_id}', '{file_name}', current_timestamp(), "{load_status}", {row_count})
        """)
    else:
        print(f"{file_name} is not JSON; run the CSV block")


In [0]:
%sql

-- CHILD SUCCESS: 
UPDATE workspace.logs.child_log
SET EndTime = current_timestamp(), Status = 'SUCCESS'
WHERE MasterLogId = (
        SELECT MasterLogId
        FROM workspace.logs.master_log
        WHERE JobName = 'Training_Job' AND Status = 'RUNNING'
        ORDER BY MasterLogId DESC
        LIMIT 1
     )
  AND NotebookName = 'Staging'
  AND Status = 'RUNNING';

